# 키오스크 연구 자동 실행

GitHub 코드 → Drive 연결 → 9개 평가 기준 실험 → 체크포인트 저장·중복 정리 → 결과 저장 순서로 실행합니다. 첫 실행은 pilot으로 검증하세요.

Colab 런타임 종료 후에는 계산도 중단됩니다. 이 노트북은 백그라운드 무제한 실행이나 연결 유지 우회를 하지 않습니다. 다시 연결한 뒤 같은 커밋과 실험 ID로 재개하세요.

## 1. GitHub 버전

공개 저장소 읽기에는 토큰이 필요 없습니다. 비공개 저장소만 Colab Secrets에 `KIOSK_GITHUB_TOKEN`을 등록하세요. 재개 시 `REVISION`에 progress.json의 git_commit을 입력하세요.

In [ ]:
import os, re, sys, subprocess, tempfile
from pathlib import Path
REPOSITORY = "phj9187/koisk-simulation" #@param {type:"string"}
REVISION = "main" #@param {type:"string"}
if not re.fullmatch(r"[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", REPOSITORY):
    raise ValueError("저장소는 owner/name 형식이어야 합니다.")
if not re.fullmatch(r"[A-Za-z0-9_./-]+", REVISION) or REVISION.startswith("-"):
    raise ValueError("브랜치 또는 커밋 해시를 입력하세요.")
if any(n.startswith(("kiosk_research", "kiosk_sim", "kiosk_cloud")) for n in sys.modules):
    raise RuntimeError("이전 코드가 로드되어 있습니다. 런타임을 다시 시작한 뒤 실행하세요.")
token = None
try:
    from google.colab import userdata
    token = userdata.get("KIOSK_GITHUB_TOKEN")
except Exception:
    pass
CODE_ROOT = Path(tempfile.mkdtemp(prefix="kiosk-research-", dir="/content"))
with tempfile.TemporaryDirectory() as auth_dir:
    helper = Path(auth_dir) / "askpass.py"
    helper.write_text("#!/usr/bin/env python3\nimport os,sys\nprint('x-access-token' if 'username' in sys.argv[1].lower() else os.environ.get('KIOSK_EPHEMERAL_TOKEN',''))\n")
    helper.chmod(0o700)
    child_env = dict(os.environ, GIT_TERMINAL_PROMPT="0", GIT_ASKPASS=str(helper))
    if token:
        child_env["KIOSK_EPHEMERAL_TOKEN"] = token
    try:
        commands = [
            ["git", "init", str(CODE_ROOT)],
            ["git", "-C", str(CODE_ROOT), "remote", "add", "origin", "https://github.com/"+REPOSITORY+".git"],
            ["git", "-C", str(CODE_ROOT), "fetch", "--depth", "1", "origin", REVISION],
            ["git", "-C", str(CODE_ROOT), "checkout", "--detach", "FETCH_HEAD"],
        ]
        for command in commands:
            result = subprocess.run(command, env=child_env, capture_output=True, text=True)
            if result.returncode:
                raise RuntimeError("GitHub 읽기 실패. 저장소·버전·비공개 저장소 권한을 확인하세요. 토큰은 출력하지 않습니다.")
    finally:
        child_env.pop("KIOSK_EPHEMERAL_TOKEN", None)
        token = None
COMMIT = subprocess.check_output(["git","-C",str(CODE_ROOT),"rev-parse","HEAD"],text=True).strip()
subprocess.check_call([sys.executable,"-m","pip","install","-q","-r",str(CODE_ROOT/"requirements.txt")])
sys.path.insert(0,str(CODE_ROOT/"src"))
print("코드 경로:", CODE_ROOT)
print("실행 커밋:", COMMIT)


## 2. Drive 연결

처음 인증 요청은 직접 승인하세요. 공유 PC에서는 본인 계정인지 확인하세요.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
DATA_ROOT = Path("/content/drive/MyDrive/AI-Kiosk-Data")
DATA_ROOT.mkdir(parents=True, exist_ok=True)


## 3. 자동 실행 또는 재개

`pilot`: 8매장·9기준·15방문자·학습2/검증1/평가1일·1반복. `research`: 8매장·9기준·30방문자·20/5/5일·5반복. 모두 가상 자료입니다. 실측 자료는 별도 검증된 설정이 필요합니다.

`RESUME_ID`를 지정하면 원래 설정을 사용합니다. 빈 ID로 재실행하면 새 실험이 생깁니다. 중지는 별도 셀에서 STOP 파일을 만들거나 실행 중단을 누를 수 있습니다. STOP은 오늘 완료 후 안전 중지, 강제 중단은 마지막 저장된 매장 부분부터 복구합니다.

RUNNING.lock이 남은 경우 이전 런타임이 종료됐음을 확인한 뒤 해당 실험의 잠금 파일만 수동 삭제하세요. 활성 런타임의 잠금을 자동으로 지우지 않습니다.

In [ ]:
PRESET = "pilot" #@param ["pilot", "research"]
RESUME_ID = "" #@param {type:"string"}
DAY_BUDGET = 0 #@param {type:"integer"}
# 0: 설정된 전체 일정. 양수: 지정 일수 후 안전 중지.
from kiosk_research.config import Config
from kiosk_research.session import run_session
config = Config.load(CODE_ROOT / "config" / ("screening_"+PRESET+".yaml"))
run = run_session(DATA_ROOT, config=config, resume_id=RESUME_ID.strip() or None, day_budget=DAY_BUDGET or None)


## 4. 진행 상황·저장 결과

이 셀은 실행 완료 후 자동으로 표시됩니다. 실행 중에는 Drive의 latest_run.json 또는 experiments/실험ID/progress.json에서 마지막 갱신 시각·완료 날짜·체크포인트·완료 매장 수를 확인하세요. 파일의 running 표시가 오래됐다면 현재 런타임이 살아 있다는 보장은 아닙니다.

In [ ]:
import json
from kiosk_research.session import status
print(json.dumps(status(DATA_ROOT), ensure_ascii=False, indent=2))
from kiosk_research.persistence import read_json
print(read_json(run.directory / "session_result.json"))


## 5. 선택: GitHub 요약 업로드

아래 설정을 true로 바꾼 경우에만 Colab Secrets의 쓰기 권한 토큰을 사용합니다. 원시 자료·체크포인트는 Drive에 남고 작은 요약만 새로운 results 브랜치에 올라갑니다. 이미 같은 요약을 올렸으면 저장된 업로드 기록을 보여줍니다.

In [ ]:
UPLOAD_SUMMARY = False #@param {type:"boolean"}
if UPLOAD_SUMMARY:
    from google.colab import userdata
    from kiosk_research.github import upload_summary
    summary = Path(read_json(run.directory / "session_result.json")["summary"])
    if (summary / "github_upload.json").exists():
        print(read_json(summary / "github_upload.json"))
    else:
        print(upload_summary(summary, REPOSITORY, userdata.get("KIOSK_GITHUB_TOKEN")))
